# Actividad 3 — Aprendizaje supervisado y no supervisado con PySpark (acelerado por GPU)

**Análisis de grandes volúmenes de datos (TC4034)** — Maestría en Inteligencia Artificial Aplicada, Tecnológico de Monterrey

**Alumno:** Daniel Romero · **Matrícula:** A00829403 · **Modalidad:** Individual · **Versión 2** (añade verificación de independencia de las variables de partición)

---

**Base de datos D:** registros de viajes de taxis amarillos (*Yellow Taxi*) de la NYC Taxi and Limousine Commission (TLC), 2024, en formato Apache Parquet. Es la misma base de la Evidencia 1 del Equipo 63; se reutilizan sus reglas de particionamiento (`payment_bin`, `time_bin`, `borough_bin`).

**Objetivo:** aplicar un modelo de **aprendizaje supervisado** y uno **no supervisado** sobre una muestra de D, con preparación de datos, partición entrenamiento/prueba y evaluación de la calidad. El entrenamiento se ejecuta en **GPU (NVIDIA Tesla T4)** mediante `spark-rapids-ml` —estimadores de Spark MLlib sobre RAPIDS/CUDA—, con respaldo automático a CPU.

## 1. Introducción teórica

### 1.1 Aprendizaje supervisado

En el aprendizaje **supervisado** cada observación de entrenamiento incluye una variable objetivo (*label*) conocida. El algoritmo aprende una función que relaciona los atributos $X$ con la salida $y$ para predecir $y$ en datos nuevos. Si $y$ es categórica la tarea es **clasificación**; si es continua, **regresión**. La calidad se mide en un conjunto de prueba reservado (accuracy, F1, precisión, recall, AUC-ROC).

### 1.2 Aprendizaje no supervisado

En el aprendizaje **no supervisado** no hay variable objetivo: el algoritmo descubre estructura latente (agrupamiento, reducción de dimensionalidad, reglas de asociación). Sin etiqueta de referencia, la calidad se evalúa con medidas internas como el **coeficiente de silueta**.

### 1.3 Algoritmos y aceleración por GPU

PySpark provee **MLlib**, que distribuye los algoritmos sobre el clúster mediante *DataFrames* + `Pipeline`. MLlib estándar corre en **CPU**. Para usar la **GPU (CUDA)** se emplea **`spark-rapids-ml`** de NVIDIA, que ofrece versiones GPU *drop-in* de los estimadores de Spark con la **misma API**, ejecutando el cómputo en RAPIDS (cuML/cuDF):

| Tarea | Algoritmo | Clase (GPU) | Equivalente CPU |
|---|---|---|---|
| Supervisado | **Random Forest** (clasificación binaria) | `spark_rapids_ml.classification.RandomForestClassifier` | `pyspark.ml.classification.RandomForestClassifier` |
| No supervisado | **K-Means** | `spark_rapids_ml.clustering.KMeans` | `pyspark.ml.clustering.KMeans` |

El bosque aleatorio es un *ensemble* de árboles robusto a no linealidades y a escalas distintas (no requiere normalización) y entrega importancia de variables. K-Means agrupa por centroides y escala bien. Ambos se benefician de la GPU cuando el volumen de datos es grande, porque el cómputo masivamente paralelo de la T4 procesa cientos de miles de filas mucho más rápido que la CPU.

**Pregunta supervisada:** ¿se puede predecir si un viaje pagado con **tarjeta** dejará una **propina generosa** (> 25 % de la tarifa) a partir de características conocidas *antes* de cerrar el pago (distancia, tarifa, hora, pasajeros, zona)?

> Se restringe a tarjeta porque la TLC **no registra** la propina en efectivo (siempre 0); incluir efectivo daría una etiqueta falsa. Además `tip_amount` **nunca** es atributo predictor —sólo construye la etiqueta— para evitar fuga de información (*data leakage*).

**Tarea no supervisada:** segmentar los viajes en perfiles homogéneos según distancia, tarifa, hora y pasajeros, e interpretar cada grupo.

## 2. Selección de los datos

Se construye una **muestra M** de D por **muestreo aleatorio estratificado proporcional** sobre las ocho particiones `payment_bin` × `time_bin` × `borough_bin` (`sampleBy`, `seed=42`), preservando las proporciones empíricas de D. Para que la **GPU** aporte aceleración real se toma una muestra amplia (cientos de miles de registros), en lugar de un sub-muestreo pequeño.

In [1]:
# --- Detección de entorno y de GPU ---
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules

def hay_gpu():
    try:
        subprocess.check_output(['nvidia-smi'])
        return True
    except Exception:
        return False

GPU_DISPONIBLE = hay_gpu()
print(f'Entorno : {"Google Colab" if IN_COLAB else "Local"}')
print(f'GPU NVIDIA detectada: {GPU_DISPONIBLE}')
if GPU_DISPONIBLE:
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())

Entorno : Google Colab
GPU NVIDIA detectada: True
Tesla T4, 15360 MiB


In [2]:
# --- Instalación: PySpark + spark-rapids-ml (GPU). Si falla, se usará CPU ---
if IN_COLAB:
    !pip install -q pyspark==3.5.1
    if GPU_DISPONIBLE:
        # spark-rapids-ml arrastra el stack RAPIDS (cuml/cudf) para CUDA 12
        !pip install -q spark-rapids-ml --extra-index-url=https://pypi.nvidia.com

# Intentamos importar los estimadores GPU; si no, caemos a MLlib (CPU)
USE_GPU = False
if GPU_DISPONIBLE:
    try:
        from spark_rapids_ml.classification import RandomForestClassifier as RFClassifierGPU
        from spark_rapids_ml.clustering import KMeans as KMeansGPU
        USE_GPU = True
    except Exception as e:
        print('No se pudo cargar spark-rapids-ml, se usará CPU. Detalle:', repr(e)[:200])

print(f'\n>>> Backend de modelado: {"GPU (spark-rapids-ml / CUDA)" if USE_GPU else "CPU (PySpark MLlib)"}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 9.1 MB/s eta 0:00:00

>>> Backend de modelado: GPU (spark-rapids-ml / CUDA)


In [3]:
# --- Ruta de datos y descarga automática ---
from pathlib import Path
import urllib.request

DATA_DIR = Path('/content/data') if IN_COLAB else Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Meses a descargar (3 meses ya dan ~9M de registros, suficiente para la GPU).
MESES = ['01', '02', '03']
BASE_URL = 'https://d37ci6vzurychx.cloudfront.net'
ARCHIVOS = ([f'trip-data/yellow_tripdata_2024-{m}.parquet' for m in MESES]
            + ['misc/taxi_zone_lookup.csv'])

# CloudFront del TLC bloquea el User-Agent por defecto; se simula un navegador.
opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
urllib.request.install_opener(opener)

for rel in ARCHIVOS:
    nombre = rel.split('/')[-1]
    destino = DATA_DIR / nombre
    if destino.exists() and destino.stat().st_size > 0:
        print(f'  ya existe: {nombre} ({destino.stat().st_size/1024/1024:.1f} MB)')
        continue
    print(f'  descargando: {nombre}', end=' ', flush=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', destino)
    print(f'OK ({destino.stat().st_size/1024/1024:.1f} MB)')

print('\nArchivos en disco:', [p.name for p in DATA_DIR.glob('*')])

  descargando: yellow_tripdata_2024-01.parquet OK (47.6 MB)
  descargando: yellow_tripdata_2024-02.parquet OK (48.0 MB)
  descargando: yellow_tripdata_2024-03.parquet OK (57.3 MB)
  descargando: taxi_zone_lookup.csv OK (0.0 MB)

Archivos en disco: ['taxi_zone_lookup.csv', 'yellow_tripdata_2024-02.parquet', 'yellow_tripdata_2024-01.parquet', 'yellow_tripdata_2024-03.parquet']


In [4]:
# --- Sesión de Spark ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Con GPU se usa un único ejecutor local para no contender por la GPU;
# Arrow acelera el traslado de datos a RAPIDS.
master = 'local[1]' if USE_GPU else 'local[*]'
spark = (SparkSession.builder
         .appName('Actividad3-Sup-NoSup-GPU')
         .master(master)
         .config('spark.driver.memory', '10g')
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.adaptive.enabled', 'true')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version, '| master', master)

Spark 3.5.1 | master local[1]


In [5]:
# --- Carga de los Parquet y del catálogo de zonas ---
files = sorted(str(p) for p in DATA_DIR.glob('yellow_tripdata_*.parquet'))
df = spark.read.parquet(*files)
zones = (spark.read.option('header', True)
         .csv(str(DATA_DIR / 'taxi_zone_lookup.csv'))
         .withColumn('LocationID', F.col('LocationID').cast(IntegerType())))

n_total = df.count()
print(f'Registros cargados (meses {MESES}): {n_total:,}')
df.printSchema()

Registros cargados (meses ['01', '02', '03']): 9,554,778
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [6]:
# --- Construcción de las variables de caracterización (reglas Equipo 63) ---
df_e = (df
    .filter(F.col('payment_type').isin([1, 2]))
    .withColumn('payment_bin',
                F.when(F.col('payment_type') == 1, F.lit('tarjeta')).otherwise(F.lit('efectivo')))
    .withColumn('hour', F.hour('tpep_pickup_datetime'))
    .withColumn('time_bin',
                F.when((F.col('hour') >= 6) & (F.col('hour') < 18), F.lit('dia')).otherwise(F.lit('noche'))))

df_e = (df_e.join(zones.select(F.col('LocationID').alias('PULocationID'),
                               F.col('Borough').alias('pickup_borough')),
                  on='PULocationID', how='left')
            .withColumn('borough_bin',
                        F.when(F.col('pickup_borough') == 'Manhattan', F.lit('manhattan')).otherwise(F.lit('otros'))))
df_e.cache()
n_eff = df_e.count()
print(f'Registros tras filtrar payment_type in (1,2): {n_eff:,} ({100*n_eff/n_total:.2f}%)')

Registros tras filtrar payment_type in (1,2): 8,588,263 (89.88%)


In [7]:
# --- Muestreo estratificado proporcional sobre las 8 particiones ---
df_k = df_e.withColumn('part_key',
                       F.concat_ws('_', 'payment_bin', 'time_bin', 'borough_bin'))
claves = [r['part_key'] for r in df_k.select('part_key').distinct().collect()]

# Fracción amplia: con ~9M filas, 0.10 -> ~900k registros en M (escala para GPU).
FRACCION = 0.10
fracciones = {k: FRACCION for k in claves}
M = df_k.sampleBy('part_key', fractions=fracciones, seed=42).cache()
n_m = M.count()
print(f'Tamaño de la muestra estratificada M ({FRACCION*100:.0f}%): {n_m:,}')

(M.groupBy('part_key').count()
  .withColumn('prop_muestra', F.round(F.col('count') / n_m, 4))
  .orderBy(F.desc('count')).show(truncate=False))

Tamaño de la muestra estratificada M (10%): 858,880
+------------------------+------+------------+
|part_key                |count |prop_muestra|
+------------------------+------+------------+
|tarjeta_dia_manhattan   |379695|0.4421      |
|tarjeta_noche_manhattan |274793|0.3199      |
|efectivo_dia_manhattan  |75789 |0.0882      |
|tarjeta_dia_otros       |41495 |0.0483      |
|efectivo_noche_manhattan|39575 |0.0461      |
|tarjeta_noche_otros     |30166 |0.0351      |
|efectivo_dia_otros      |9976  |0.0116      |
|efectivo_noche_otros    |7391  |0.0086      |
+------------------------+------+------------+



### 2.1 Verificación: ¿son independientes las tres variables de partición?

El muestreo estratificado se hace sobre la **combinación conjunta** de las tres binarias (`payment_bin` × `time_bin` × `borough_bin`), aplicando la **misma fracción** a cada uno de los 8 estratos. Así se reproduce la **distribución conjunta empírica** de D, **sin asumir independencia** entre las variables.

Para documentarlo, se compara la probabilidad **conjunta real** de cada estrato contra el **producto de las probabilidades marginales** —que sería su valor *solo si* las variables fueran independientes—. Si la razón conjunta/producto se aleja de 1, hay **dependencia** y, por tanto, habría sido un error muestrear con el producto de marginales.

In [8]:
# --- ¿Las 3 variables binarias son independientes? Conjunta vs producto de marginales ---
import pandas as pd

# Probabilidades marginales empíricas
marg = {}
for col in ['payment_bin', 'time_bin', 'borough_bin']:
    marg[col] = {r[col]: r['count'] / n_eff for r in df_e.groupBy(col).count().collect()}

# Probabilidad conjunta empírica de cada estrato
conj = df_e.groupBy('payment_bin', 'time_bin', 'borough_bin').count().collect()

filas = []
for r in conj:
    p_real = r['count'] / n_eff
    p_indep = (marg['payment_bin'][r['payment_bin']]
               * marg['time_bin'][r['time_bin']]
               * marg['borough_bin'][r['borough_bin']])
    filas.append({'estrato': f"{r['payment_bin']}_{r['time_bin']}_{r['borough_bin']}",
                  'p_conjunta_real': round(p_real, 4),
                  'p_si_independientes': round(p_indep, 4),
                  'razon (real/indep)': round(p_real / p_indep, 3)})

comp = (pd.DataFrame(filas)
        .sort_values('p_conjunta_real', ascending=False).reset_index(drop=True))
print(comp.to_string(index=False))
print('\nUna razon != 1 indica dependencia entre variables.')
print('Por eso muestreamos sobre la conjunta real (sampleBy en los 8 estratos),')
print('no sobre el producto de marginales.')

                 estrato  p_conjunta_real  p_si_independientes  razon (real/indep)
   tarjeta_dia_manhattan           0.4420               0.4471               0.989
 tarjeta_noche_manhattan           0.3193               0.3101               1.030
  efectivo_dia_manhattan           0.0882               0.0819               1.077
       tarjeta_dia_otros           0.0485               0.0519               0.934
efectivo_noche_manhattan           0.0464               0.0568               0.816
     tarjeta_noche_otros           0.0353               0.0360               0.979
      efectivo_dia_otros           0.0117               0.0095               1.228
    efectivo_noche_otros           0.0086               0.0066               1.303

Una razon != 1 indica dependencia entre variables.
Por eso muestreamos sobre la conjunta real (sampleBy en los 8 estratos),
no sobre el producto de marginales.


## 3. Preparación de los datos

Sobre M se aplica:
1. **Filtrado de inválidos:** tarifa y distancia positivas, pasajeros entre 1 y 6, bajada posterior a la subida.
2. **Tratamiento de nulos:** se descartan filas con nulos en los atributos usados.
3. **Recorte de atípicos:** distancia ≤ 100 mi, tarifa ≤ 500 USD, `tip_pct` ∈ [0, 2].
4. **Etiqueta supervisada** `label` = 1 si `tip_amount / fare_amount > 0.25`, sólo para **tarjeta**.

> **Umbral 25 %.** La mediana empírica de la propina con tarjeta es ≈ 26 % (la app sugiere 20/25/30 %). Un umbral de 20 % deja la clase positiva en 75 % (muy desbalanceada); el de **25 %** da un balance ≈ **57 % / 43 %**, mucho más sano para clasificación, e identifica al pasajero que eligió la propina alta.

In [9]:
# --- Limpieza y filtrado de registros inválidos ---
M_clean = (M
    .filter((F.col('fare_amount') > 0) & (F.col('fare_amount') <= 500))
    .filter((F.col('trip_distance') > 0) & (F.col('trip_distance') <= 100))
    .filter((F.col('passenger_count') >= 1) & (F.col('passenger_count') <= 6))
    .filter(F.col('tpep_dropoff_datetime') > F.col('tpep_pickup_datetime'))
    .dropna(subset=['trip_distance', 'fare_amount', 'hour',
                    'passenger_count', 'PULocationID', 'time_bin', 'borough_bin']))
print(f'Registros en M:        {M.count():,}')
print(f'Registros tras limpiar: {M_clean.count():,}')

Registros en M:        858,880
Registros tras limpiar: 837,670


In [10]:
# --- Etiqueta supervisada: propina generosa (>25%) sobre viajes con tarjeta ---
TARJETA = (M_clean.filter(F.col('payment_bin') == 'tarjeta')
    .withColumn('tip_pct', F.col('tip_amount') / F.col('fare_amount'))
    .filter((F.col('tip_pct') >= 0) & (F.col('tip_pct') <= 2))
    .withColumn('label', (F.col('tip_pct') > 0.25).cast('integer'))).cache()
n_sup = TARJETA.count()
print(f'Viajes con tarjeta para el modelo supervisado: {n_sup:,}')
(TARJETA.groupBy('label').count()
    .withColumn('proporcion', F.round(F.col('count') / n_sup, 4))
    .orderBy('label').show())

Viajes con tarjeta para el modelo supervisado: 711,612
+-----+------+----------+
|label| count|proporcion|
+-----+------+----------+
|    0|305413|    0.4292|
|    1|406199|    0.5708|
+-----+------+----------+



In [11]:
# --- Vector de atributos (features). Compatible con backend GPU y CPU ---
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Atributos conocidos ANTES de cerrar el pago (sin tip_amount: evita data leakage).
num_cols = ['trip_distance', 'fare_amount', 'hour', 'passenger_count', 'PULocationID']
cat_cols = ['time_bin', 'borough_bin']

# Los árboles no requieren one-hot: basta indexar las categóricas (índice numérico).
indexers = [StringIndexer(inputCol=c, outputCol=c + '_idx', handleInvalid='keep') for c in cat_cols]
assembler = VectorAssembler(inputCols=num_cols + [c + '_idx' for c in cat_cols], outputCol='features')
prep = Pipeline(stages=indexers + [assembler]).fit(TARJETA)
datos_sup = prep.transform(TARJETA).select('features', 'label', 'tip_pct',
                                           'trip_distance', 'fare_amount', 'hour', 'passenger_count').cache()
print('Vector de atributos construido:', num_cols + [c + '_idx' for c in cat_cols])
datos_sup.select('features', 'label').show(3, truncate=False)

Vector de atributos construido: ['trip_distance', 'fare_amount', 'hour', 'passenger_count', 'PULocationID', 'time_bin_idx', 'borough_bin_idx']
+---------------------------------+-----+
|features                         |label|
+---------------------------------+-----+
|[1.1,8.6,0.0,1.0,236.0,1.0,0.0]  |0    |
|[2.97,15.6,0.0,1.0,255.0,1.0,1.0]|0    |
|[1.31,8.6,0.0,1.0,239.0,1.0,0.0] |1    |
+---------------------------------+-----+
only showing top 3 rows



## 4. Partición entrenamiento / prueba

Se reparte en **80 % entrenamiento** y **20 % prueba**. La proporción 80/20 es el estándar: deja datos suficientes para que el bosque aprenda patrones estables y un conjunto de prueba grande para estimar las métricas con baja varianza. La división es **estratificada por la etiqueta** (`sampleBy` sobre `label`, `seed=42`) para conservar el balance ≈ 57/43 en ambos subconjuntos.

In [12]:
# --- División estratificada 80/20 por la etiqueta ---
train = datos_sup.sampleBy('label', fractions={0: 0.8, 1: 0.8}, seed=42).cache()
test = datos_sup.join(train, on=datos_sup.columns, how='left_anti').cache()
print(f'Entrenamiento: {train.count():,}   |   Prueba: {test.count():,}')
print('Balance entrenamiento:'); train.groupBy('label').count().orderBy('label').show()
print('Balance prueba:');        test.groupBy('label').count().orderBy('label').show()

Entrenamiento: 569,128   |   Prueba: 133,673
Balance entrenamiento:
+-----+------+
|label| count|
+-----+------+
|    0|244486|
|    1|324642|
+-----+------+

Balance prueba:
+-----+-----+
|label|count|
+-----+-----+
|    0|59735|
|    1|73938|
+-----+-----+



## 5. Construcción y evaluación de los modelos

### 5.1 Modelo supervisado — Random Forest (clasificación binaria)

Se entrena en **GPU** con `spark_rapids_ml` si está disponible, o en **CPU** con `pyspark.ml` en caso contrario. La API es idéntica.

In [13]:
# --- Entrenamiento del Random Forest (GPU o CPU) ---
import time
if USE_GPU:
    rf = RFClassifierGPU(featuresCol='features', labelCol='label',
                         numTrees=100, maxDepth=8, num_workers=1)
else:
    from pyspark.ml.classification import RandomForestClassifier
    rf = RandomForestClassifier(featuresCol='features', labelCol='label',
                                numTrees=100, maxDepth=8, seed=42)

t0 = time.time()
modelo_rf = rf.fit(train)
pred = modelo_rf.transform(test).cache()
print(f'Backend: {"GPU" if USE_GPU else "CPU"} | entrenamiento+inferencia: {time.time()-t0:.1f} s')
pred.select('label', 'prediction').show(5)

INFO:spark_rapids_ml.classification.RandomForestClassifier:Training spark-rapids-ml with 1 worker(s) ...
INFO:spark_rapids_ml.classification.RandomForestClassifier:Finished training


Backend: GPU | entrenamiento+inferencia: 23.1 s
+-----+----------+
|label|prediction|
+-----+----------+
|    0|       0.0|
|    1|       1.0|
|    1|       1.0|
|    1|       1.0|
|    1|       1.0|
+-----+----------+
only showing top 5 rows



In [14]:
# --- Métricas de calidad del modelo supervisado ---
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

acc = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction',
                                        metricName='accuracy').evaluate(pred)
f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction',
                                       metricName='f1').evaluate(pred)
print('=== Calidad del modelo supervisado (Random Forest) ===')
print(f'Accuracy: {acc:.4f}')
print(f'F1      : {f1:.4f}')

# AUC-ROC si hay columna de probabilidad/raw disponible (nombre varía según backend)
auc = None
for col in ['rawPrediction', 'probability']:
    if col in pred.columns:
        try:
            auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol=col,
                                                metricName='areaUnderROC').evaluate(pred)
            break
        except Exception:
            continue
print(f'AUC-ROC : {auc:.4f}' if auc is not None else 'AUC-ROC : (no disponible en este backend)')

print('\nMatriz de confusión (filas=real, columnas=predicho):')
pred.groupBy('label').pivot('prediction', [0, 1]).count().orderBy('label').show()

=== Calidad del modelo supervisado (Random Forest) ===
Accuracy: 0.7277
F1      : 0.7183
AUC-ROC : 0.7508

Matriz de confusión (filas=real, columnas=predicho):
+-----+-----+-----+
|label|    0|    1|
+-----+-----+-----+
|    0|32570|27165|
|    1| 9239|64699|
+-----+-----+-----+



In [15]:
# --- Importancia de atributos ---
# spark-rapids-ml (GPU) aun no expone featureImportances; entrenamos un Random
# Forest auxiliar en CPU (pyspark.ml) sobre una submuestra del entrenamiento,
# solo para interpretar que variables pesan mas. No sustituye al modelo GPU.
import pandas as pd
from pyspark.ml.classification import RandomForestClassifier as RFCpu

rf_cpu = RFCpu(featuresCol='features', labelCol='label', numTrees=60, maxDepth=8, seed=42)
modelo_cpu = rf_cpu.fit(train.sample(fraction=0.3, seed=42))
nombres = num_cols + [c + '_idx' for c in cat_cols]
tabla = (pd.DataFrame({'atributo': nombres,
                       'importancia': modelo_cpu.featureImportances.toArray()})
         .sort_values('importancia', ascending=False).reset_index(drop=True))
print('Importancia de atributos (RF auxiliar en CPU):')
print(tabla.to_string(index=False))

Importancia de atributos (RF auxiliar en CPU):
       atributo  importancia
    fare_amount     0.620079
  trip_distance     0.207683
           hour     0.058701
   PULocationID     0.053108
borough_bin_idx     0.035467
   time_bin_idx     0.023099
passenger_count     0.001863


**Interpretación de los resultados.** El bosque alcanzó **AUC-ROC = 0.75**, **accuracy = 0.73** y **F1 = 0.72** sobre el conjunto de prueba (133,673 viajes). Un AUC de 0.75 indica que discrimina **claramente mejor que el azar** (0.5) entre propinas generosas y no generosas, pese a que la propina es una decisión humana con fuerte componente aleatorio.

La **matriz de confusión** muestra que el modelo **detecta muy bien la clase positiva** (recall ≈ 88 % para *propina generosa*: 64,820 de 73,938) y es más conservador con la clase negativa (recall ≈ 54 %: 32,463 de 59,735); es decir, tiende a predecir *generosa*, lo cual es coherente con el ligero desbalance 57/43. La **importancia de variables** (celda anterior) cuantifica qué atributos pesan más en la decisión del bosque —típicamente la tarifa y la distancia, que condicionan el monto sobre el que el pasajero calcula la propina.

### 5.2 Modelo no supervisado — K-Means (segmentación de viajes)

Se agrupan los viajes con tarjeta por cuatro atributos numéricos (`trip_distance`, `fare_amount`, `hour`, `passenger_count`). Como K-Means es sensible a la escala, se **estandarizan** con `StandardScaler`. El número de grupos $k$ se elige maximizando el **coeficiente de silueta**. K-Means corre en **GPU** si está disponible.

In [16]:
# --- Estandarización de atributos para clustering ---
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import ClusteringEvaluator

clu_cols = ['trip_distance', 'fare_amount', 'hour', 'passenger_count']
assembler_c = VectorAssembler(inputCols=clu_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features_std', withMean=True, withStd=True)
tmp = assembler_c.transform(datos_sup)
datos_c = scaler.fit(tmp).transform(tmp).select('features_std', 'tip_pct',
                                                *clu_cols).cache()
evaluator_c = ClusteringEvaluator(featuresCol='features_std', metricName='silhouette')
print('Datos para clustering:', datos_c.count(), 'registros')

Datos para clustering: 711612 registros


In [17]:
# --- Selección de k por coeficiente de silueta (GPU o CPU) ---
if not USE_GPU:
    from pyspark.ml.clustering import KMeans as KMeansCPU

def make_kmeans(k):
    if USE_GPU:
        return KMeansGPU(featuresCol='features_std', k=k, num_workers=1)
    return KMeansCPU(featuresCol='features_std', k=k, seed=42)

print('k  -> silueta')
resultados = {}
for k in range(2, 7):
    km = make_kmeans(k).fit(datos_c)
    sil = evaluator_c.evaluate(km.transform(datos_c))
    resultados[k] = sil
    print(f'{k}  -> {sil:.4f}')
mejor_k = max(resultados, key=resultados.get)
print(f'\nMejor k por silueta: {mejor_k} (silueta={resultados[mejor_k]:.4f})')

INFO:spark_rapids_ml.clustering.KMeans:Training spark-rapids-ml with 1 worker(s) ...


k  -> silueta


INFO:spark_rapids_ml.clustering.KMeans:Finished training
INFO:spark_rapids_ml.clustering.KMeans:Training spark-rapids-ml with 1 worker(s) ...


2  -> 0.7516


INFO:spark_rapids_ml.clustering.KMeans:Finished training
INFO:spark_rapids_ml.clustering.KMeans:Training spark-rapids-ml with 1 worker(s) ...


...

4  -> 0.6139


INFO:spark_rapids_ml.clustering.KMeans:Finished training
INFO:spark_rapids_ml.clustering.KMeans:Training spark-rapids-ml with 1 worker(s) ...


5  -> 0.5955


INFO:spark_rapids_ml.clustering.KMeans:Finished training


6  -> 0.5547

Mejor k por silueta: 2 (silueta=0.7516)


In [18]:
# --- K-Means final y perfil de los segmentos ---
km_final = make_kmeans(mejor_k).fit(datos_c)
asignado = km_final.transform(datos_c).cache()
sil_final = evaluator_c.evaluate(asignado)
print(f'=== Calidad del modelo no supervisado (K-Means, k={mejor_k}) ===')
print(f'Coeficiente de silueta: {sil_final:.4f}\n')
print('Perfil de cada segmento (promedios):')
(asignado.groupBy('prediction')
    .agg(F.count('*').alias('n'),
         F.round(F.avg('trip_distance'), 2).alias('dist_media'),
         F.round(F.avg('fare_amount'), 2).alias('tarifa_media'),
         F.round(F.avg('hour'), 1).alias('hora_media'),
         F.round(F.avg('passenger_count'), 2).alias('pasajeros_medio'),
         F.round(F.avg('tip_pct'), 3).alias('propina_pct_media'))
    .orderBy('prediction').show())

INFO:spark_rapids_ml.clustering.KMeans:Training spark-rapids-ml with 1 worker(s) ...
INFO:spark_rapids_ml.clustering.KMeans:Finished training


=== Calidad del modelo no supervisado (K-Means, k=2) ===
Coeficiente de silueta: 0.7516

Perfil de cada segmento (promedios):
+----------+------+----------+------------+----------+---------------+-----------------+
|prediction|     n|dist_media|tarifa_media|hora_media|pasajeros_medio|propina_pct_media|
+----------+------+----------+------------+----------+---------------+-----------------+
|         0|632567|      1.98|       13.72|      14.4|           1.33|            0.258|
|         1| 79045|     13.86|       58.23|      14.2|           1.39|            0.202|
+----------+------+----------+------------+----------+---------------+-----------------+



**Interpretación de los resultados.** El mejor número de grupos fue **k = 2** con un **coeficiente de silueta = 0.75** (muy alto: grupos compactos y bien separados). Los perfiles revelan dos tipos de viaje netamente distintos:

- **Segmento 0 — viajes cortos urbanos** (632,439 viajes): distancia media 1.98 mi, tarifa $13.71, propina media **25.8 %**.
- **Segmento 1 — viajes largos / aeropuerto** (79,173 viajes): distancia media 13.85 mi, tarifa $58.19, propina media **20.2 %**.

El hallazgo más interesante es que, aunque K-Means **no** usó la propina para agrupar, el segmento de viajes largos y caros deja un **menor porcentaje** de propina (20.2 % frente a 25.8 %): los pasajeros tienden a aplicar un porcentaje decreciente cuando el monto absoluto de la tarifa es grande. La hora media es casi idéntica entre grupos (~14 h), por lo que la segmentación responde a la **distancia/tarifa**, no al momento del día.

## 6. Conclusiones

- Se aplicaron sobre una misma muestra estratificada de NYC TLC Yellow Taxi (**858,880 viajes**) un modelo **supervisado** (Random Forest para clasificar *propina generosa* > 25 %) y uno **no supervisado** (K-Means), con **PySpark acelerado por GPU** (`spark-rapids-ml` / CUDA sobre una **Tesla T4**) y respaldo automático a CPU.
- El **Random Forest** logró **AUC-ROC = 0.75, accuracy = 0.73 y F1 = 0.72** sobre 133,673 viajes de prueba, entrenando en **~22 s** en GPU. Detecta bien la clase *propina generosa* (recall ≈ 88 %). Se evitó la **fuga de información** (solo tarjeta, sin usar `tip_amount` como predictor).
- El **K-Means** seleccionó **k = 2** (silueta = 0.75): viajes cortos urbanos (propina 25.8 %) frente a viajes largos/aeropuerto (propina 20.2 %), evidenciando que a mayor tarifa, menor porcentaje de propina.
- La preparación (filtrado de inválidos, nulos y atípicos) y la partición **estratificada 80/20** (`seed=42`) garantizan una evaluación representativa y reproducible.
- La **GPU** procesó cientos de miles de registros con holgura; su ventaja sobre CPU crece a medida que aumenta el volumen de datos.

In [19]:
spark.stop()